# Trip-chain browser

Walk through the persons whose trip chain carries a given `problemas` code (a defect left) or `ajustes`
code (a change `load_eod` made), one person at a time, with the chain **as shipped** next to what
**`load_eod()` returns**. Every table is read live through `load_eod`, so the browser reflects the current
rules after any change to `eodgdl.eod`.

**How to use.** Pick codes in the two lists (Ctrl/Cmd-click for several; `dropped` selects the persons who
lost a duplicate return). *any selected* keeps a person if one of their rows carries any chosen code;
*all selected* keeps a person only if their chain carries every chosen code. Step with **◀ ▶**, jump with
the counter or **random**, or type `household,person` in *go to*. Rows that carry a selected code are
highlighted; a cell that the rules changed reads `shipped → cleaned`; a dropped row is struck through.
In the timeline, grey bars are returns home, blue bars activities, each running from the start time
across the leg minutes; a hatched bar was edited or imputed, a red edge carries a `problemas` code.

Notes typed under a person are saved to `notebooks/chain_browser_notes.json` (or the working directory
when `EODGDL_DATA_DIR` points elsewhere); *with a note only* filters to them, so a pattern can be
collected across sessions.

In [1]:
import json
import os
import random
from pathlib import Path

import ipywidgets as w
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display


def _find_data_dir() -> Path:
    marker = "IMEPLAN_Base_Viviendas_Master.csv"
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / "data"
        if (cand / marker).exists():
            return cand.resolve()
    raise FileNotFoundError("Could not locate the eodgdl data/ directory.")


os.environ.setdefault("EODGDL_DATA_DIR", str(_find_data_dir()))

from eodgdl import load_eod
from eodgdl.eod import FIX_CODES, FIX_FLAG, ISSUE_CODES, ISSUE_FLAG, has_code

P = ["folio_vivienda", "folio_habitante"]
HOME_MOTIVE = "Regresar a Casa"
DATA_DIR = Path(os.environ["EODGDL_DATA_DIR"])
NOTES_PATH = (DATA_DIR.parent / "notebooks" if (DATA_DIR.parent / "notebooks").is_dir() else Path.cwd()) / "chain_browser_notes.json"

raw = load_eod(clean_chains=False)
clean = load_eod()
R, C = raw.trips, clean.trips
Cr = C.reindex(R.index)                      # the cleaned row for every shipped row; NaN where it was dropped
home = raw.viv.ageb.astype(str)
legmin = raw.legs.groupby(level=[0, 1, 2]).traslado_min.sum().reindex(R.index).fillna(0).astype(int)


def minutes(df):
    return df.hora_inicio_h.astype(float) * 60 + df.hora_inicio_m.astype(float)


rows = pd.DataFrame({
    "start_shipped": minutes(R), "start_clean": minutes(Cr),
    "motive_shipped": R.motivo_viaje.astype(object), "motive_clean": Cr.motivo_viaje.astype(object),
    "type_shipped": R.tipo_lugar_destino.astype(object), "type_clean": Cr.tipo_lugar_destino.astype(object),
    "origin_shipped": R.origen.astype(str), "origin_clean": Cr.origen.astype(object),
    "destination": R.destino.astype(str), "mode": R.modo_principal.astype(str), "leg_min": legmin,
    "ajustes": Cr[FIX_FLAG].fillna(""), "problemas": Cr[ISSUE_FLAG].fillna(""),
    "dropped": Cr[FIX_FLAG].isna(),
}, index=R.index)
rows["home"] = home.reindex(rows.index.get_level_values(0)).to_numpy()

# one boolean column per code, computed once so the filters are instant
code_cols = {c: has_code(rows.problemas, c) for c in ISSUE_CODES}
code_cols["dropped"] = rows.dropped
code_cols.update({c: has_code(rows.ajustes, c) for c in FIX_CODES})
codes = pd.DataFrame(code_cols)


def codes_of(s):
    return frozenset(c for v in s for c in v.split(";") if c)


by_person = rows.groupby(level=P)
persons = pd.DataFrame({
    "n_trips": by_person.size(),
    "fix_codes": by_person.ajustes.agg(codes_of),
    "issue_codes": by_person.problemas.agg(codes_of),
    "dropped": by_person.dropped.any(),
}).join(clean.hab[["sexo_nacimiento", "edad", "ocupacion", "viajes_contados", "diario_repetido"]])
persons["home"] = home.reindex(persons.index.get_level_values(0)).to_numpy()
person_codes = codes.groupby(level=P).any()

notes = json.loads(NOTES_PATH.read_text()) if NOTES_PATH.exists() else {}

summary = pd.DataFrame({
    "column": ["problemas"] * (len(ISSUE_CODES) + 1) + ["ajustes"] * len(FIX_CODES),
    "code": list(ISSUE_CODES) + ["dropped"] + list(FIX_CODES),
})
summary["rows"] = [int(codes[c].sum()) for c in summary.code]
summary["persons"] = [int(person_codes[c].sum()) for c in summary.code]
print(f"{len(rows):,} shipped trip rows, {len(persons):,} persons with trips; notes file: {NOTES_PATH}")
summary.style.hide(axis="index")

154,662 shipped trip rows, 52,758 persons with trips; notes file: /Users/gperaza/Research/eodgdl/notebooks/chain_browser_notes.json


column,code,rows,persons
problemas,regreso_en_casa,491,409
problemas,hora_invertida,885,767
problemas,origen_discontinuo,32,32
problemas,regreso_sin_llegar,163,153
problemas,tipo_destino_dudoso,16,16
problemas,hora_nocturna,93,93
problemas,hora_anterior,25,25
problemas,hora_repetida,101,101
problemas,hora_traslapada,569,550
problemas,inicio_fuera_de_casa,906,906


In [2]:
def hhmm(m):
    return "—" if pd.isna(m) else f"{int(m) // 60:02d}:{int(m) % 60:02d}"


def arrow(a, b):
    return a if a == b else f"{a} → {b}"


def person_frame(key, matching):
    # the person's rows as a display table, shipped → cleaned where the rules changed a value
    t = rows.loc[key]
    h = persons.loc[key, "home"]
    zone = lambda x: "—" if pd.isna(x) else ("home" if str(x) == h else str(x))
    text = lambda x: "—" if pd.isna(x) else str(x)
    out = []
    for fv, r in t.iterrows():
        dropped = bool(r.dropped)
        out.append({
            "trip": fv,
            "start": hhmm(r.start_shipped) if dropped else arrow(hhmm(r.start_shipped), hhmm(r.start_clean)),
            "motive": text(r.motive_shipped) if dropped else arrow(text(r.motive_shipped), text(r.motive_clean)),
            "dest. type": text(r.type_shipped) if dropped else arrow(text(r.type_shipped), text(r.type_clean)),
            "origin": zone(r.origin_shipped) if dropped else arrow(zone(r.origin_shipped), zone(r.origin_clean)),
            "destination": zone(r.destination),
            "mode": r["mode"], "leg min": int(r.leg_min),
            "ajustes": r.ajustes, "problemas": r.problemas,
            "status": "dropped" if dropped else ("changed" if r.ajustes else ""),
            "_match": bool(matching.get((key[0], key[1], fv), False)),
        })
    return pd.DataFrame(out)


def style_frame(df):
    def row_css(r):
        if r["status"] == "dropped":
            return ["color:#999; text-decoration:line-through"] * len(r)
        if r["_match"]:
            return ["background-color:#fff3b0"] * len(r)
        return [""] * len(r)
    return (df.style.apply(row_css, axis=1).hide(axis="index").hide(axis="columns", subset=["_match"])
            .set_table_styles([{"selector": "th, td", "props": "padding:2px 8px; font-size:12px; text-align:left"}]))


def header_html(key):
    p = persons.loc[key]
    codes_txt = ", ".join(sorted(p.issue_codes | p.fix_codes | ({"dropped"} if p.dropped else set()))) or "clean"
    rep = " · <b>repeated diary</b>" if p.diario_repetido else ""
    return (f"<h4 style='margin:4px 0'>household {key[0]}, person {key[1]}</h4>"
            f"<div style='font-size:12px'>home zone {p.home} · {p.sexo_nacimiento}, {int(p.edad)} · {p.ocupacion if pd.notna(p.ocupacion) else '—'} · "
            f"{int(p.viajes_contados)} trips counted, {int(p.n_trips)} rows shipped{rep}<br>codes: {codes_txt}</div>")


def timeline(key):
    t = rows.loc[key]
    fig, ax = plt.subplots(figsize=(10, 2.3))
    for lane, col in enumerate(["start_clean", "start_shipped"]):
        motive_col = "motive_clean" if col == "start_clean" else "motive_shipped"
        for fv, r in t.iterrows():
            s = r[col]
            if pd.isna(s):
                continue
            color = "#969696" if r[motive_col] == HOME_MOTIVE else "#2c7fb8"
            flagged = col == "start_clean" and bool(r.problemas)
            edited = col == "start_clean" and bool(r.ajustes)
            ax.broken_barh([(s / 60, max(int(r.leg_min), 4) / 60)], (lane - 0.35, 0.7), facecolors=color,
                           edgecolors="#e6550d" if flagged else color, linewidth=1.8, hatch="///" if edited else None)
            ax.text(s / 60, lane + 0.38, str(fv), fontsize=8, ha="left", va="bottom")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["after load_eod", "as shipped"])
    ax.set_ylim(-0.7, 1.9)
    ax.set_xlim(0, 24)
    ax.set_xticks(range(0, 25, 2))
    ax.set_xlabel("hour of day")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()


# ---- widgets
issue_sel = w.SelectMultiple(options=list(ISSUE_CODES) + ["dropped"], rows=9, description="problemas",
                             layout=w.Layout(width="320px"))
fix_sel = w.SelectMultiple(options=list(FIX_CODES), rows=9, description="ajustes", layout=w.Layout(width="320px"))
mode_sel = w.ToggleButtons(options=["any selected", "all selected"], value="any selected", description="match")
min_trips = w.IntSlider(min=1, max=int(persons.n_trips.max()), value=1, description="min trips", continuous_update=False)
diary_sel = w.Dropdown(options=[("any diary", None), ("repeated diary", True), ("not repeated", False)], value=None,
                       description="diary")
noted_only = w.Checkbox(value=False, description="with a note only", indent=False)
prev_btn, next_btn, rand_btn = w.Button(description="◀"), w.Button(description="▶"), w.Button(description="random")
pos = w.BoundedIntText(min=1, max=1, value=1, description="#", layout=w.Layout(width="160px"))
count_lbl = w.HTML()
goto = w.Text(placeholder="household,person", description="go to", layout=w.Layout(width="260px"))
note_box = w.Textarea(placeholder="a note about this person's chain…", layout=w.Layout(width="640px", height="60px"))
save_btn = w.Button(description="save note")
note_lbl = w.HTML()
out = w.Output()

state = {"keys": [], "i": 0, "matching": pd.Series(False, index=rows.index), "busy": False}


def selected_codes():
    return list(issue_sel.value), list(fix_sel.value)


def filtered_keys():
    issues, fixes = selected_codes()
    chosen = issues + fixes
    if chosen:
        matching = codes[chosen].any(axis=1)
        keep = person_codes[chosen].all(axis=1) if mode_sel.value == "all selected" else person_codes[chosen].any(axis=1)
    else:
        matching = pd.Series(False, index=rows.index)
        keep = pd.Series(True, index=persons.index)
    keep &= persons.n_trips >= min_trips.value
    if diary_sel.value is not None:
        keep &= persons.diario_repetido == diary_sel.value
    if noted_only.value:
        keep &= pd.Series([f"{k[0]}/{k[1]}" in notes for k in persons.index], index=persons.index)
    return [tuple(k) for k in persons.index[keep]], matching


def render():
    keys = state["keys"]
    with out:
        out.clear_output(wait=True)
        if not keys:
            display(HTML("<i>no person matches the selection</i>"))
            return
        key = keys[state["i"]]
        display(HTML(header_html(key)))
        display(style_frame(person_frame(key, state["matching"])))
        timeline(key)
    note_box.value = notes.get(f"{key[0]}/{key[1]}", "") if keys else ""
    count_lbl.value = f"<b>{state['i'] + 1:,} / {len(keys):,}</b> persons" if keys else "<b>0</b> persons"


def move_to(i):
    if not state["keys"]:
        return
    state["i"] = int(i) % len(state["keys"])
    state["busy"] = True
    pos.value = state["i"] + 1
    state["busy"] = False
    render()


def refresh(_=None):
    current = state["keys"][state["i"]] if state["keys"] else None
    state["keys"], state["matching"] = filtered_keys()
    pos.max = max(len(state["keys"]), 1)
    move_to(state["keys"].index(current) if current in state["keys"] else 0)


def on_pos(change):
    if not state["busy"]:
        move_to(change["new"] - 1)


def on_goto(_):
    try:
        hh, person = (int(x) for x in goto.value.replace(" ", "").split(","))
    except ValueError:
        return
    key = (hh, person)
    if key not in persons.index:
        count_lbl.value = f"<i>no trips for household {hh}, person {person}</i>"
        return
    if key not in state["keys"]:
        state["keys"] = [key]
        pos.max = 1
    move_to(state["keys"].index(key))


def on_save(_):
    if not state["keys"]:
        return
    key = state["keys"][state["i"]]
    k = f"{key[0]}/{key[1]}"
    if note_box.value.strip():
        notes[k] = note_box.value.strip()
    else:
        notes.pop(k, None)
    NOTES_PATH.write_text(json.dumps(notes, indent=2, ensure_ascii=False))
    note_lbl.value = f"<span style='font-size:12px'>{len(notes)} notes saved to {NOTES_PATH.name}</span>"


for widget in (issue_sel, fix_sel, mode_sel, min_trips, diary_sel, noted_only):
    widget.observe(refresh, "value")
pos.observe(on_pos, "value")
prev_btn.on_click(lambda _: move_to(state["i"] - 1))
next_btn.on_click(lambda _: move_to(state["i"] + 1))
rand_btn.on_click(lambda _: move_to(random.randrange(len(state["keys"]))) if state["keys"] else None)
goto.continuous_update = False          # fires on Enter or focus loss
goto.observe(on_goto, "value")
save_btn.on_click(on_save)

ui = w.VBox([
    w.HBox([issue_sel, fix_sel, w.VBox([mode_sel, min_trips, diary_sel, noted_only])]),
    w.HBox([prev_btn, next_btn, pos, count_lbl, rand_btn, goto]),
    out,
    w.HBox([note_box, w.VBox([save_btn, note_lbl])]),
])
refresh()
ui

## Starting points

- `hora_invertida` with `hora:vecinos` under *all selected*: the imputed rows the typo search could not
  rescue, to check that the fault sits in the neighbours' times.
- `origen_discontinuo` alone: the 32 zone breaks left, to look for which side is wrong.
- `regreso_sin_llegar` against `tipo_destino_dudoso`: returns that did not reach home versus returns that
  did with a doubtful type, the two sides of the recode's zone guard.
- `hora:-10h+12h`: the compound two-hour reading, the most expensive edit in the menu, to validate it holds
  up chain by chain.
- `inicio_fuera_de_casa` with `fin_fuera_de_casa`: days away from home at both ends, night shifts and stays
  elsewhere.